In [1]:
import os
import shutil
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [6]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/postnauka.csv',
)

dataset.get_possible_modalities()

{'@2gramm', '@3gramm', '@author', '@post_tag', '@snippet', '@title', '@word'}

In [7]:
MAIN_MODALITY = '@word'

In [8]:
dataset._data.head()

,id,vw_text,raw_text
id,,,
1.txt,1.txt,1.txt |@author fuchs preobrazhensky tabachniko...,@title Автограф # «Математический дивертисмент...
2.txt,2.txt,2.txt |@word книга:2 лекция:3 рассматриваться:...,@title Главы: Маскулинности в российском конте...
3.txt,3.txt,3.txt |@word развитие появляться пиджина:4 бел...,@title Пиджины и креольские языки | @snippet Л...
4.txt,4.txt,4.txt |@word стандартный задача:3 состоять:4 р...,@title FAQ: Физиология микроводорослей | @snip...
5.txt,5.txt,5.txt |@2gramm повседневный_практика государст...,@title Русская государственная идеология | @sn...


In [9]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [10]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 4.21 s, sys: 166 ms, total: 4.37 s
Wall time: 4.32 s


In [11]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [12]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [13]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [57]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, topic_names: List[str], parent_model=None, parent_phi=None):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._parent_model = parent_model
        self._parent_phi = parent_phi

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)

        if self._parent_phi is not None:
            parent_phi = self._parent_phi
            vals = parent_phi.values
        else:
            parent_phi = self._parent_model.get_phi()
            vals = parent_phi.values[:, self._topic_indices]

        assert vals.shape[0] == rwt.shape[0]
        assert vals.shape[1] == len(self._topic_indices), (vals.shape[1], len(self._topic_indices))
        
        rwt[:, self._topic_indices] += vals

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [15]:
class DecorrelatorWithOtherPhiRegularizer(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._other_topic_sum = self._other_phi.values.sum(
            axis=1, keepdims=True
        )
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        # print('Decorring')

        rwt = np.zeros_like(pwt)
        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * self._other_topic_sum
        )

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [16]:
class DecorrelatorWithOtherPhiRegularizer2(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi, num_iters: Optional[int] = None):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._num_iters = num_iters
        self._cur_iter = 0
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        rwt = np.zeros_like(pwt)
        
        if self._num_iters is not None and self._cur_iter >= self._num_iters:
            return rwt

        correlations = cdist(
            self._other_phi.values.T,
            pwt.values[:, self._topic_indices].T,
            lambda u, v: (u * v).sum()
        )
        weighted_other_topics = self._other_phi.values.dot(correlations)

        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * weighted_other_topics
        )
        self._cur_iter += 1

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [17]:
NUM_TOPICS = 50  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [18]:
NUM_GOOD_TOPICS_THRESHOLD = NUM_TOPICS - 5

In [19]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [20]:
# HIGH_COHERENCE_THRESHOLD = 1.0014147388051453
# LOW_COHERENCE_THRESHOLD = 0.49633189795078303

def is_good(coherence):
    return 1.0703095408390997 <= coherence

def is_bad(coherence):
    return coherence <= 0.6300154717967157

In [21]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [22]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [23]:
TOPIC_INDICES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

In [24]:
# New
#  Best: 100000.0 55.10831705729197
# Close: 10000 55.840738932291515

In [25]:
# New
#  Best: 100000000.0 55.63102213541697
# Close: 10000000.0 55.890380859375

In [24]:
MAX_NUM_TRAINS

20

In [25]:
BEST_TAUS = [100000, 100000000]

In [26]:
import json
import os

SAVE_FOLDER = 'results50/postnauka'

os.makedirs(SAVE_FOLDER, exist_ok=True)

In [59]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

DECORRELATION_TAUS = [BEST_TAUS[0]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    
    res_file_path = SAVE_FOLDER + f'/iterative_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv')

            raise NotImplementedError
        # else:
        #     good_phi = None
        #     bad_phi = None
        

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

            good_phi = phi[good_topic_names]
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
    
    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.07116935463855091
sparse_theta_sp: -0.4011325611325611
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 8, 'not_good': 40, 'total_bad': 8}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253ad0aeb0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2530c88550>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2530c88a00>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.08896169329818866
sparse_theta_sp: -0.5014157014157014
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 24, 'bad': 4, 'not_good': 26, 'total_bad': 12}
Removing: results50/postnauka/iterative_100000/0
2
test_1
test_2
test_3
test_4
test_5


...

test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253bdb6b50>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f253bdb6b80>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f253ad0a520>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.1368641435356749
sparse_theta_sp: -0.7714087714087713
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 31, 'bad': 2, 'not_good': 19, 'total_bad': 14}
Removing: results50/postnauka/iterative_100000/1
3
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253bdb6c10>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f253bdb6bb0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2544f81640>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.1872877753646077
sparse_theta_sp: -1.055612002980424
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 37, 'bad': 1, 'not_good': 13, 'total_bad': 15}
Removing: results50/postnauka/iterative_100000/2
4
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2544a39a30>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2530cb9c10>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2544a39040>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.2737282870713498
sparse_theta_sp: -1.5428175428175426
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
SUCCESS: no bad topics!
num_topics: {'good': 46, 'bad': 0, 'not_good': 4, 'total_bad': 15}
Removing: results50/postnauka/iterative_100000/3


In [58]:
! ls results50/postnauka/iterative_100000

ls: cannot access 'results50/postnauka/iterative_100000': No such file or directory


In [50]:
! ls results50/postnauka

ablation_study		      _iterative_100000.json	 plsa_with_cohs.json
bertopic		      iterative2_100000000	 sparse.json
bertopic.json		      iterative2_100000000.json  sparse_with_cohs.json
decorrelation.json	      lda.json			 tless.json
decorrelation_with_cohs.json  lda_with_cohs.json	 tless_with_cohs.json
_iterative_100000	      plsa.json


In [33]:
results.keys()

dict_keys([])

In [36]:
SAVE_FOLDER

'results50/postnauka'

In [210]:
! ls $SAVE_FOLDER

decorrelation.json     iterative_10000.json  plsa.json	  tless.json
iterative_100000.json  lda.json		     sparse.json


In [60]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

DECORRELATION_TAUS = [BEST_TAUS[1]]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau

    res_file_path = SAVE_FOLDER + f'/iterative2_{int(key)}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue
    
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        
        seed_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'iterative2_{key}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            raise NotImplementedError()
        # else:
        #     good_phi = None
        #     bad_phi = None
        

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

            good_phi = phi[good_topic_names]
            
            del result, phi
            model = None

        else:

            print('test_1')

            if good_phi is None:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
            else:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_phi=good_phi,
                    topic_names=good_topic_names,
                )

            print('test_2')
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            print('test_3')
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )

            print('test_4')
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )

            print('test_5')
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            print('test_6')
            
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }

            print('test_7')
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            print('test_8')
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            del prev_model
            
            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

            

            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    for k, r in results.items():
        for s in r:
            s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
    
    for k, r in results.items():
        with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

100000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.07116935463855091
sparse_theta_sp: -0.4011325611325611
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 8, 'not_good': 40, 'total_bad': 8}
1
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2530c88550>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f2544f61bb0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253a954f10>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.08896169329818866
sparse_theta_sp: -0.5014157014157014
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 18, 'bad': 3, 'not_good': 32, 'total_bad': 11}
Removing: results50/postnauka/iterative2_100000000/0
2
test_1
test_2
test_3
test_4
test_5


...

test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2530c884f0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253bc6b250>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253b160490>}
Skipping computation of dists: (1081, 1225).
Skipping computation of dists: (1081, 1225).
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.5930779553212578
sparse_theta_sp: -3.342771342771343
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
num_topics: {'good': 44, 'bad': 5, 'not_good': 6, 'total_bad': 60}
Removing: results50/postnauka/iterative2_100000000/16
18
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253af44fd0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253ba97160>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253ad0a490>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.5930779553212578
sparse_theta_sp: -3.342771342771343
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
SUCCESS: less bad topics
num_topics: {'good': 44, 'bad': 4, 'not_good': 6, 'total_bad': 64}
Removing: results50/postnauka/iterative2_100000000/17
19
test_1
test_2
test_3
test_4
test_5


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



test_6
test_7
{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253b840be0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f25681d66d0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f253b9bb040>}
test_8
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.5930779553212578
sparse_theta_sp: -3.342771342771343
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000000
ext_decorr_good: 100000000
DOWNFALL: more bad topics...
num_topics: {'good': 44, 'bad': 5, 'not_good': 6, 'total_bad': 69}
Removing: results50/postnauka/iterative2_100000000/18


In [35]:
results.keys()

dict_keys([])

In [36]:
! ls $SAVE_FOLDER

ablation_study	    iterative_100000.json      lda.json     tless.json
decorrelation.json  iterative2_100000000       plsa.json
iterative_100000    iterative2_100000000.json  sparse.json


## Ablation Study

In [30]:
DECORRELATION_TAU = BEST_TAUS[0]

ALL_PARAMS = [
    # (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    # (0, 1, 0),
    # (0, 0, 1),
]

In [31]:
ALL_PARAMS = [(1, 0, 1)]  # re-run

In [32]:
SAVE_FOLDER + f'/ablation_study'

'results50/postnauka/ablation_study'

In [33]:
os.makedirs(SAVE_FOLDER + f'/ablation_study', exist_ok=True)

In [40]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.07116935463855091
sparse_theta_sp: -0.4011325611325611
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 8, 'not_good': 40, 'total_bad': 8}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2544884490>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2544a36fd0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.08896169329818866
sparse_theta_sp: -0.5014157014157014
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 18, 'bad': 6, 'not_good': 32, 'total_bad': 14}
Removing: results50/postnauka/ablation_study/iterative_100000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2533f93d60>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2533af1580>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.5083525331325066
sparse_theta_sp: -2.8652325795182936
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000
num_topics: {'good': 43, 'bad': 7, 'not_good': 7, 'total_bad': 55}
Removing: results50/postnauka/ablation_study/iterative_100000_1-0-1/10
12
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f253aaa18e0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2544eb8d00>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.5083525331325066
sparse_theta_sp: -2.8652325795182936
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000
num_topics: {'good': 43, 'bad': 7, 'not_good': 7, 'total_bad': 62}
Removing: results50/postnauka/ablation_study/iterative_100000_1-0-1/11
13
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f2544eb8bb0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f2533fd8460>}
Skipping computation of dists: (1081, 1225).
Skipping computation of dists: (1081, 1225).
Early stopping because of corrupted topics
Saving results


In [36]:
! mv results50/postnauka/ablation_study/iterative_100000_1-0-1.json results50/postnauka/ablation_study/_iterative_100000_1-0-1.json

In [43]:
! ls results50/postnauka/ablation_study

iterative_100000_1-0-0	      iterative_100000_1-1-0.json
iterative_100000_1-0-0.json   iterative2_100000000_1-0-0
_iterative_100000_1-0-1       iterative2_100000000_1-0-0.json
iterative_100000_1-0-1	      iterative2_100000000_1-0-1
_iterative_100000_1-0-1.json  iterative2_100000000_1-0-1.json
iterative_100000_1-0-1.json   iterative2_100000000_1-1-0
iterative_100000_1-1-0	      iterative2_100000000_1-1-0.json


In [41]:
! ls results50/postnauka/ablation_study/_iterative_100000_1-0-1

0  1  10  11  12  2  3	4  5  6  7  8  9


In [42]:
! ls results50/postnauka/ablation_study/iterative_100000_1-0-1

0  1  10  11  12  2  3	4  5  6  7  8  9


In [45]:
! cat results50/postnauka/ablation_study/iterative_100000_1-0-1.json

[
    {
        "scores": {
            "perplexity": 2734.291259765625,
            "coherence_20": 0.8857382699236708,
            "diversity_euclidean": 0.07294457211078517,
            "diversity_jensenshannon": 0.68663828544553,
            "diversity_hellinger": 0.8041930224472627,
            "diversity_cosine": 0.8546272571913327
        },
        "topic_coherences": {
            "0": 1.1009096854534886,
            "1": 0.841738661947743,
            "2": 1.0609399512231388,
            "3": 1.1579246220975674,
            "4": 1.252173499731989,
            "5": 1.0065504981528222,
            "6": 0.7979344145757231,
            "7": 0.642265826629241,
            "8": 1.2138650467715681,
            "9": 0.9531445204347976,
            "10": 1.2809597883297656,
            "11": 0.5745370479464954,
            "12": 0.978982771129228,
            "13": 0.8756547514790309,
            "14": 0.9417386909975013,
            "15": 0.9168161194848754,
            "16": 1.35907

In [38]:
! mv results50/postnauka/ablation_study/iterative_100000_1-0-1 results50/postnauka/ablation_study/_iterative_100000_1-0-1

In [47]:
DECORRELATION_TAU = BEST_TAUS[1]

In [48]:
DECORRELATION_TAU

100000000

In [49]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

for params in ALL_PARAMS:
    key = params

    output_k = '-'.join(str(i) for i in key)
    res_file_path = SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json'

    if os.path.isfile(res_file_path):
        print(f'Already trained: {res_file_path}. Skipping')
        continue

    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    good_phi = None
    seed = 0

    os.makedirs(
        os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}'),
        exist_ok=True
    )

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_GOOD_TOPICS_THRESHOLD:
        print(seed)

        seed_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed))
        prev_save_folder = os.path.join(SAVE_FOLDER, f'ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}', str(seed - 1))

        if os.path.isdir(seed_save_folder):
            print(f'Loading seed results from "{seed_save_folder}".')
            
            good_phi = pd.read_csv(f'{seed_save_folder}/good_phi.csv', index_col=0)
            bad_phi = pd.read_csv(f'{seed_save_folder}/bad_phi.csv', index_col=0)

            with open(f'{seed_save_folder}/topic_names.json', 'r') as f:
                topic_names = json.loads(f.read())

            with open(f'{seed_save_folder}/results.json', 'r') as f:
                results[key] = json.loads(f.read())

            print(f'Loaded result: {results[key]}.')
            
            good_topic_names = topic_names['good']
            bad_topic_names = topic_names['bad']
            not_good_topic_names = topic_names['not_good']

            seed += 1

            continue

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])
            
            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )
            
            
            del result, phi
            model = None

        else:
            # assert False
            
            custom_regularizers = dict()
            
            if params[0]:
                if good_phi is None:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_model=prev_model._model,
                        topic_names=good_topic_names,
                    )
                else:
                    fix_regularizer = FastFixPhiRegularizer(
                        name='fix',
                        parent_phi=good_phi,
                        topic_names=good_topic_names,
                    )
    
                print('test_2')
                
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            # cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]
                bad_phi = cur_bad_phi
            else:
                pass
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                # Done at the end of iterration
                # bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            if params[1]:
                bad_phi = deepcopy(bad_phi)
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None

            if good_phi is None:
                good_phi = prev_model._model.get_phi()[good_topic_names]
    
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)

            if new_result['scores']['diversity_jensenshannon'] == -1:
                print('Early stopping because of corrupted topics')

                break
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1


            
            os.makedirs(seed_save_folder)

            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)

            good_phi = prev_model._model.get_phi()[good_topic_names]

            good_phi.to_csv(f'{seed_save_folder}/good_phi.csv')
            bad_phi.to_csv(f'{seed_save_folder}/bad_phi.csv')

            with open(f'{seed_save_folder}/topic_names.json', 'w') as f:
                f.write(
                    json.dumps(
                        {
                            'good': good_topic_names,
                            'bad': bad_topic_names,
                            'not_good': not_good_topic_names,
                        }
                    )
                )

            for k, r in results.items():
                for s in r:
                    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

            with open(f'{seed_save_folder}/results.json', 'w') as f:
                f.write(
                    json.dumps(
                        results[key]
                    )
                )

            print(f'Removing: {prev_save_folder}')
            # shutil.rmtree(prev_save_folder)

    print('Saving results')

    for k, r in results.items():
        output_k = '-'.join(str(i) for i in k)
        with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
            f.write(
                json.dumps(r, indent=4)
            )

(1, 0, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.07116935463855091
sparse_theta_sp: -0.4011325611325611
decorrelation: 0.01
None
num_topics: {'good': 10, 'bad': 8, 'not_good': 40, 'total_bad': 8}
1
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6e45088400>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f6e45522640>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.08896169329818866
sparse_theta_sp: -0.5014157014157014
decorrelation: 0.01
fix: 1000000000
ext_decorr_good: 100000000
SUCCESS: more good topics
num_topics: {'good': 17, 'bad': 8, 'not_good': 33, 'total_bad': 16}
Removing: results50/postnauka/ablation_study/iterative2_100000000_1-0-1/0
2
test_2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6e45633130>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.15471598834467593
sparse_theta_sp: -0.8720273068099155
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 27, 'bad': 7, 'not_good': 23, 'total_bad': 140}
Removing: results50/postnauka/ablation_study/iterative2_100000000_1-0-0/16
18
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6e439cb3d0>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.15471598834467593
sparse_theta_sp: -0.8720273068099155
decorrelation: 0.01
fix: 1000000000
SUCCESS: less bad topics
num_topics: {'good': 27, 'bad': 6, 'not_good': 23, 'total_bad': 146}
Removing: results50/postnauka/ablation_study/iterative2_100000000_1-0-0/17
19
test_2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f6d87989850>}
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.15471598834467593
sparse_theta_sp: -0.8720273068099155
decorrelation: 0.01
fix: 1000000000
DOWNFALL: more bad topics...
num_topics: {'good': 27, 'bad': 9, 'not_good': 23, 'total_bad': 155}
Removing: results50/postnauka/ablation_study/iterative2_100000000_1-0-0/18
Saving results


In [51]:
1

1

In [53]:
results.keys()

dict_keys([(1, 0, 1), (1, 1, 0), (1, 0, 0)])